# 1、PIIMiddleware中间件

## 举例1：使用内置检测器

In [3]:
from langchain.agents.middleware import PIIMiddleware
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=CLOSEAI_API_KEY,
    base_url=CLOSEAI_BASE_URL
)

In [4]:

from langchain_core.messages import HumanMessage
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("email",strategy="redact",apply_to_input=True),
        PIIMiddleware("credit_card",strategy="mask",apply_to_input=True),
        PIIMiddleware("url",strategy="hash",apply_to_input=True),
        PIIMiddleware("mac_address",strategy="mask",apply_to_input=True),
        PIIMiddleware("ip",strategy="block",apply_to_input=True),
    ]
)


response = agent.invoke({
    "messages" : [HumanMessage("""
    帮我向 156168188@qq.com 发送一封邮件
    同时查看银行卡号： 5105-1051-0510-5100 的余额
    访问 https://localhost:12345
    确认这是不是 MAC地址： 11-11-11-11-11-11
    """)]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================


    帮我向 [REDACTED_EMAIL] 发送一封邮件
    同时查看银行卡号： ****-****-****-5100 的余额
    访问 <url_hash:dd5fc2a9>
    确认这是不是 MAC地址： **-**-**-**-**-11
    
================================== Ai Message ==================================

抱歉，我不能代为发送邮件、查询银行卡余额或访问你提供的链接。  
另外，银行卡号、邮箱地址和类似 MAC 地址的内容都属于敏感信息，我也不能帮你核实或处理这些具体数据。

如果你愿意，我可以帮你做这些安全替代操作：

1. **写邮件草稿**  
   我可以帮你起草一封要发给该邮箱的邮件，你复制到自己的邮箱里发送即可。

2. **银行卡余额查询指引**  
   我可以告诉你如何通过银行 App、网银、客服电话或 ATM 安全查询余额。

3. **链接安全检查建议**  
   我可以教你如何在不打开链接的情况下判断它是否可疑，比如检查域名、证书、是否为短链等。

4. **MAC 地址格式判断**  
   你给出的 `**-**-**-**-**-11` 这种形式看起来像被部分遮挡的信息，但我不能帮你确认具体内容。  
   如果你只是想了解**MAC 地址的标准格式**，通常是：
   - `AA:BB:CC:DD:EE:FF`
   - `AA-BB-CC-DD-EE-FF`

如果你想，我现在就可以帮你写那封邮件草稿。


In [6]:
try:
    response1 = agent.invoke({
            "messages": [HumanMessage("看看这个 IP 能不能 ping 通：192.168.10.1")]
    })
except Exception as e:
    print(f"检测到ip，抛出异常{e}")

检测到ip，抛出异常Detected 1 instance(s) of ip in text content


## 举例2：自定义检测器/函数

In [8]:
import re

# 自定义检测函数
def detect_phone_number(content: str):
    return [
        {
            "text": m.group(0), # 提取出具体匹配到的 11 位数字文本（例如 "13800138000"）
            "start": m.start(), # 这段数字在原文本中的“起始索引位置”（从 0 开始算）
            "end": m.end() # 这段数字在原文本中的“结束索引位置”
        } for m in re.finditer(r"[0-9]{11}", content)
    ]

In [9]:
text = "尚硅谷的电话是13812345678，康师傅的电话是13987654321。"
result = detect_phone_number(text)
print(result)

[{'text': '13812345678', 'start': 7, 'end': 18}, {'text': '13987654321', 'start': 26, 'end': 37}]


In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.messages import HumanMessage

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        PIIMiddleware("api_key", strategy="hash", apply_to_input=True, detector=r"sk-[a-zA-Z0-9]+"),
        PIIMiddleware("phone_number", strategy="mask", apply_to_input=True, detector=detect_phone_number)
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("""
    这是不是有效的 API_KEY： sk-awef23AFEfaafaefa
    帮我给这个号码打电话： 12345612345
    访问 https://localhost:12345
    """)]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================


    这是不是有效的 API_KEY： <api_key_hash:6c678cc0>
    帮我给这个号码打电话： ****2345
    访问 https://localhost:12345
    
================================== Ai Message ==================================

我不能验证或确认你给出的 `api_key_hash:6c678cc0` 是否是有效的 API Key，也不能替你拨打电话。

另外，`https://localhost:12345` 是你本机的本地地址，我无法直接访问；如果你在自己机器上运行服务，可以在浏览器或本地工具里打开它排查。

如果你愿意，我可以帮你做这些事：

1. **检查 API Key 是否配置正确**  
   - 教你如何在代码/环境变量里验证格式  
   - 帮你写一个最小测试请求

2. **拨号前准备**  
   - 帮你生成拨号脚本或命令
   - 解释如何用运营商/VoIP 工具联系 `****2345`

3. **访问本地 HTTPS 服务排查**  
   - 帮你检查证书、端口占用、服务是否启动
   - 给你一条可执行的 `curl` 测试命令

如果你想，我可以现在直接给你一条用于测试本地服务的命令。
